# Model Benchmarks: Bayesian-Value, Bayesian-Belief & Baselines

Compare predictive models of human role choice against baselines.

**Models:**
- **Bayesian-Value**: Softmax over expected values given posterior beliefs (fka "Bayesian finetuned")
- **Bayesian-Belief**: Marginalize posterior directly for role predictions (fka "Posterior Sampling")
- 7 baselines: Random, Random Walk, Optimal, Top-K, Random-to-Optimal, Copy Others, Contradict Others

**Data:** March 6 + March 18 exports (human rounds only).

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    softmax_role_dist, combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds and teacher_forced_predictions
# (these use the team-round dict format with env_config attached)
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

## 1. Load Data

Load team-round records (human rounds only, with env configs) using `online_model_sim.load_team_rounds`.
We point it at our shared data directories.

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## 2. Helper Functions

In [3]:
def run_baseline(records, predict_fn):
    """Wrap a stage-level baseline predict_fn into the format run_predictions expects.
    
    predict_fn(record, stage_idx, prev_combo, env_config) -> dict[combo_str -> prob]
    """
    def wrapped(record):
        preds = []
        for s, human_combo in enumerate(record['stage_roles']):
            prev = record['stage_roles'][s - 1] if s > 0 else None
            dist = predict_fn(record, s, prev, record['env_config'])
            marg = np.zeros(3)
            for combo, prob in dist.items():
                for c in combo:
                    marg[ROLE_CHAR_TO_IDX[c]] += prob
            marg /= 3.0
            preds.append({
                'predicted_dist': dist,
                'human_combo': human_combo,
                'model_marginal': marg,
            })
        return preds
    return run_predictions(records, wrapped)

## 3. Define Models

In [4]:
# --- Baseline 1: Random ---
def predict_random(record, stage_idx, prev_combo, env_config):
    return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}


# --- Baseline 2: Random Walk ---
def make_random_walk(eps=0.38):
    def predict(record, stage_idx, prev_combo, env_config):
        if prev_combo is None:
            return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
        dist = {}
        for combo in ALL_ROLE_COMBOS:
            p = 1.0
            for i, (c, prev_c) in enumerate(zip(combo, prev_combo)):
                if c == prev_c:
                    p *= (1.0 - eps)
                else:
                    p *= (eps / 2.0)
            dist[combo] = p
        total = sum(dist.values())
        return {c: p / total for c, p in dist.items()}
    return predict


# --- Baseline 3: Optimal ---
def predict_optimal(record, stage_idx, prev_combo, env_config):
    values = env_config['values']
    lds = record['lds']
    turn_idx = stage_idx * TURNS_PER_STAGE
    intent = lds[turn_idx] if turn_idx < len(lds) else 0
    team_hp = int(env_config['team_max_hp'])
    enemy_hp = int(env_config['enemy_max_hp'])
    
    vals = []
    for combo in ALL_ROLE_COMBOS:
        idx = ROLE_CHAR_TO_IDX[combo[0]] * 9 + ROLE_CHAR_TO_IDX[combo[1]] * 3 + ROLE_CHAR_TO_IDX[combo[2]]
        vals.append(float(values[idx, intent, min(team_hp, values.shape[2]-1), min(enemy_hp, values.shape[3]-1)]))
    max_val = max(vals)
    optimal_combos = [c for c, v in zip(ALL_ROLE_COMBOS, vals) if abs(v - max_val) < 1e-8]
    dist = {c: 0.0 for c in ALL_ROLE_COMBOS}
    for c in optimal_combos:
        dist[c] = 1.0 / len(optimal_combos)
    return dist


# --- Baseline 4: Top-K ---
def make_topk(k=7):
    def predict(record, stage_idx, prev_combo, env_config):
        values = env_config['values']
        lds = record['lds']
        turn_idx = stage_idx * TURNS_PER_STAGE
        intent = lds[turn_idx] if turn_idx < len(lds) else 0
        team_hp = int(env_config['team_max_hp'])
        enemy_hp = int(env_config['enemy_max_hp'])
        
        combo_vals = []
        for combo in ALL_ROLE_COMBOS:
            idx = ROLE_CHAR_TO_IDX[combo[0]] * 9 + ROLE_CHAR_TO_IDX[combo[1]] * 3 + ROLE_CHAR_TO_IDX[combo[2]]
            combo_vals.append((combo, float(values[idx, intent, min(team_hp, values.shape[2]-1), min(enemy_hp, values.shape[3]-1)])))
        combo_vals.sort(key=lambda x: -x[1])
        top = combo_vals[:k]
        dist = {c: 0.0 for c in ALL_ROLE_COMBOS}
        for c, v in top:
            dist[c] = 1.0 / k
        return dist
    return predict


# --- Baseline 5: Random-to-Optimal ---
def predict_random_to_optimal(record, stage_idx, prev_combo, env_config):
    alpha = stage_idx / max(len(record['stage_roles']) - 1, 1)
    random_dist = {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
    optimal_dist = predict_optimal(record, stage_idx, prev_combo, env_config)
    return {c: (1 - alpha) * random_dist[c] + alpha * optimal_dist[c] for c in ALL_ROLE_COMBOS}


# --- Baseline 6: Copy Others ---
def predict_copy_others(record, stage_idx, prev_combo, env_config):
    if prev_combo is None:
        return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
    dist = {}
    for combo in ALL_ROLE_COMBOS:
        p = 1.0
        for i in range(3):
            others = [prev_combo[j] for j in range(3) if j != i]
            if combo[i] in others:
                p *= 0.5
            else:
                p *= 0.0 if combo[i] not in others else 0.5
        dist[combo] = p
    total = sum(dist.values())
    if total > 0:
        return {c: p / total for c, p in dist.items()}
    return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}


# --- Baseline 7: Contradict Others ---
def predict_contradict(record, stage_idx, prev_combo, env_config):
    if prev_combo is None:
        return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
    dist = {}
    for combo in ALL_ROLE_COMBOS:
        p = 1.0
        for i in range(3):
            others = set(prev_combo[j] for j in range(3) if j != i)
            if combo[i] not in others:
                p *= 1.0
            else:
                p *= 0.1  # small prob for contradiction failure
        dist[combo] = p
    total = sum(dist.values())
    return {c: p / total for c, p in dist.items()}

In [5]:
# --- Bayesian-Belief (fka Posterior Sampling) ---
def make_bayesian_belief(tau_prior=2.0, epsilon=0.5):
    """Predict roles by marginalizing the joint posterior directly."""
    def predict_fn(record):
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']
        
        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        
        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break
            
            # Marginalize posterior to get per-player distributions
            per_agent = []
            for i in range(3):
                marg = np.sum(prior, axis=tuple(j for j in range(3) if j != i))
                total = marg.sum()
                per_agent.append(marg / total if total > 0 else np.ones(3) / 3)
            
            # Build joint predicted dist from marginals
            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])
            
            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })
            
            # Advance game with teacher forcing
            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp) for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions, player_stats, boss_damage, team_max_hp)
                turn_idx += 1
        
        return results
    return predict_fn


def run_bayesian_belief(records, tau_prior=2.0, epsilon=0.5):
    """Run bayesian-belief model using shared.evaluation.run_predictions."""
    return run_predictions(records, make_bayesian_belief(tau_prior, epsilon))

## 4. Run All Models

In [6]:
all_models = []

# --- Bayesian-Value (fka Bayesian finetuned) ---
# Uses oms.run_all_predictions which internally calls teacher_forced_predictions
print('Running Bayesian-Value...')
bv_results = oms.run_all_predictions(records, tau_prior=5.6, tau_softmax=0.1, epsilon=0.2)
bv_corrs = compute_pearson(bv_results)
bv_metrics = extract_metrics(bv_corrs)
all_models.append(('Bayesian-Value', bv_metrics, bv_corrs, bv_results))
print(f"  combo_r={bv_metrics['combo_r']:.4f}, marg_r={bv_metrics['marg_r']:.4f}")

# --- Bayesian-Belief (fka Posterior Sampling) ---
print('Running Bayesian-Belief...')
bb_results = run_bayesian_belief(records, tau_prior=2.0, epsilon=0.5)
bb_corrs = compute_pearson(bb_results)
bb_metrics = extract_metrics(bb_corrs)
all_models.append(('Bayesian-Belief', bb_metrics, bb_corrs, bb_results))
print(f"  combo_r={bb_metrics['combo_r']:.4f}, marg_r={bb_metrics['marg_r']:.4f}")

# --- Baselines ---
baselines = [
    ('Random', predict_random),
    ('Random Walk (eps=0.38)', make_random_walk(0.38)),
    ('Optimal', predict_optimal),
    ('Top-7', make_topk(7)),
    ('Random-to-Optimal', predict_random_to_optimal),
    ('Copy Others', predict_copy_others),
    ('Contradict Others', predict_contradict),
]

for name, fn in baselines:
    print(f'Running {name}...')
    results = run_baseline(records, fn)
    corrs = compute_pearson(results)
    metrics = extract_metrics(corrs)
    all_models.append((name, metrics, corrs, results))
    print(f"  combo_r={metrics['combo_r']:.4f}, marg_r={metrics['marg_r']:.4f}")

Running Bayesian-Value...
  combo_r=0.3970, marg_r=0.4823
Running Bayesian-Belief...
  combo_r=0.4831, marg_r=0.4053
Running Random...
  combo_r=0.1694, marg_r=nan
Running Random Walk (eps=0.38)...
  combo_r=0.4322, marg_r=0.5275
Running Optimal...
  combo_r=0.3097, marg_r=0.5025
Running Top-7...
  combo_r=0.3568, marg_r=0.4549
Running Random-to-Optimal...
  combo_r=0.2652, marg_r=0.4596
Running Copy Others...
  combo_r=0.0556, marg_r=0.5275
Running Contradict Others...
  combo_r=0.0965, marg_r=-0.5479


/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:126: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(combo_m, combo_h)
/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:129: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(marg_m, marg_h)
/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:143: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(global_marg_m, global_marg_h)
/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


## 5. Global Summary Table

In [7]:
summary_rows = []
for name, metrics, corrs, results in all_models:
    summary_rows.append({
        'Model': name,
        'combo_r': metrics['combo_r'],
        'marg_r': metrics['marg_r'],
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df = summary_df.sort_values('combo_r', ascending=False)

# Highlight best
print('\n=== Global Model Comparison ===')
print(summary_df.to_string(float_format='{:.4f}'.format))
print(f'\nBest combo_r: {summary_df.combo_r.idxmax()} ({summary_df.combo_r.max():.4f})')
print(f'Best marg_r:  {summary_df.marg_r.idxmax()} ({summary_df.marg_r.max():.4f})')


=== Global Model Comparison ===
                        combo_r  marg_r
Model                                  
Bayesian-Belief          0.4831  0.4053
Random Walk (eps=0.38)   0.4322  0.5275
Bayesian-Value           0.3970  0.4823
Top-7                    0.3568  0.4549
Optimal                  0.3097  0.5025
Random-to-Optimal        0.2652  0.4596
Random                   0.1694     NaN
Contradict Others        0.0965 -0.5479
Copy Others              0.0556  0.5275

Best combo_r: Bayesian-Belief (0.4831)
Best marg_r:  Copy Others (0.5275)


## 6. Per-Environment Comparison

In [8]:
# Build per-env table with marg_r for each model
env_ids = sorted(set(r['env_id'] for r in records))
short_names = {
    'Bayesian-Value': 'BV',
    'Bayesian-Belief': 'BB',
    'Random': 'Rand',
    'Random Walk (eps=0.38)': 'RWalk',
    'Optimal': 'Opt',
    'Top-7': 'Top7',
    'Random-to-Optimal': 'R→Opt',
    'Copy Others': 'Copy',
    'Contradict Others': 'Contra',
}

env_rows = []
for env_id in env_ids:
    row = {'env_id': env_id}
    for name, metrics, corrs, results in all_models:
        sn = short_names.get(name, name)
        env_corr = corrs.get(env_id, {})
        row[sn] = env_corr.get('marginal', {}).get('r', float('nan'))
    env_rows.append(row)

# Add global row
global_row = {'env_id': 'GLOBAL'}
for name, metrics, corrs, results in all_models:
    sn = short_names.get(name, name)
    global_row[sn] = metrics['marg_r']
env_rows.append(global_row)

env_df = pd.DataFrame(env_rows).set_index('env_id')
print('\n=== Per-Environment marg_r ===')
print(env_df.to_string(float_format='{:.3f}'.format))


=== Per-Environment marg_r ===
                    BV     BB  Rand  RWalk   Opt   Top7  R→Opt  Copy  Contra
env_id                                                                      
114_222_222_MFF  0.710  0.261   NaN  0.036 0.793  0.727  0.696 0.036  -0.119
222_222_222_FFF  0.786  0.716   NaN  0.579 0.763  0.805  0.480 0.579  -0.579
411_141_114_FFM  0.728  0.797   NaN  0.788 0.676  0.866  0.653 0.788  -0.844
411_141_114_FTF  0.791 -0.242   NaN  0.674 0.739  0.705  0.906 0.674  -0.814
411_141_114_FTM  0.294  0.804   NaN  0.790 0.000 -0.005 -0.003 0.790  -0.761
411_222_222_FMM -0.075  0.510   NaN  0.420 0.155  0.317  0.082 0.420  -0.469
GLOBAL           0.482  0.405   NaN  0.527 0.502  0.455  0.460 0.527  -0.548


## 7. Switch-Stage Analysis

Re-run models only on stages where at least one player changed roles.

In [9]:
# Filter to switch stages only
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)
        
        # Re-aggregate
        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0
        
        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])
        
        if max_stages == 0:
            continue
        
        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n
        
        filtered[env_id] = dict(data)  # copy
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })
    
    return filtered


print('\n=== Switch-Stage Performance ===')
switch_rows = []
for name, metrics, corrs, results in all_models:
    sw_results = filter_switch_stages(results)
    if sw_results:
        sw_corrs = compute_pearson(sw_results)
        sw_metrics = extract_metrics(sw_corrs)
    else:
        sw_metrics = {'combo_r': float('nan'), 'marg_r': float('nan')}
    switch_rows.append({'Model': name, 'combo_r': sw_metrics['combo_r'], 'marg_r': sw_metrics['marg_r']})

switch_df = pd.DataFrame(switch_rows).set_index('Model')
switch_df = switch_df.sort_values('combo_r', ascending=False)
print(switch_df.to_string(float_format='{:.4f}'.format))


=== Switch-Stage Performance ===
                        combo_r  marg_r
Model                                  
Bayesian-Belief          0.3043  0.5032
Bayesian-Value           0.2663  0.3390
Random-to-Optimal        0.2292  0.4274
Optimal                  0.2254  0.4003
Top-7                    0.1821  0.3574
Random Walk (eps=0.38)   0.1816  0.3588
Random                   0.1204     NaN
Copy Others              0.0430  0.3588
Contradict Others       -0.0227 -0.4346


/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:126: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(combo_m, combo_h)
/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:129: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(marg_m, marg_h)
/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:143: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(global_marg_m, global_marg_h)
/Users/jolow/coding/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


## 8. Bayesian-Belief Parameter Sweep

In [10]:
tau_values = [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 10.0, 20.0]
eps_values = [1e-10, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5]

sweep_results = []
best_combo_r = -1
best_params = None

for tau in tau_values:
    for eps in eps_values:
        results = run_bayesian_belief(records, tau_prior=tau, epsilon=eps)
        corrs = compute_pearson(results)
        m = extract_metrics(corrs)
        sweep_results.append({'tau_prior': tau, 'epsilon': eps, **m})
        if m['combo_r'] > best_combo_r:
            best_combo_r = m['combo_r']
            best_params = (tau, eps)

print(f'Best Bayesian-Belief params: tau_prior={best_params[0]}, epsilon={best_params[1]}')
print(f'Best combo_r: {best_combo_r:.4f}')

# Show as heatmap table
sweep_df = pd.DataFrame(sweep_results)
pivot = sweep_df.pivot(index='tau_prior', columns='epsilon', values='combo_r')
print('\n=== Bayesian-Belief combo_r sweep ===')
print(pivot.to_string(float_format='{:.3f}'.format))

Best Bayesian-Belief params: tau_prior=2.0, epsilon=0.5
Best combo_r: 0.4831

=== Bayesian-Belief combo_r sweep ===
epsilon    1.000000e-10  1.000000e-02  5.000000e-02  1.000000e-01  2.000000e-01  3.000000e-01  5.000000e-01
tau_prior                                                                                                  
0.5               0.392         0.413         0.419         0.424         0.423         0.415         0.363
1.0               0.394         0.402         0.421         0.432         0.448         0.458         0.432
2.0               0.362         0.366         0.381         0.399         0.427         0.450         0.483
3.0               0.331         0.335         0.349         0.365         0.395         0.422         0.475
5.0               0.303         0.306         0.318         0.333         0.362         0.389         0.449
8.0               0.287         0.290         0.301         0.315         0.342         0.367         0.425
10.0              0.